Feature 2: Power Quality Alert System
Problem: Poor power quality (voltage unbalance, low power factor, neutral current) can
damage equipment and increase costs. We need to flag anomalies.
Alert Thresholds:
1. Voltage Unbalance &gt; 2% — Warning; &gt; 5% — Critical
2. Power Factor &lt; 0.9 — Warning; &lt; 0.85 — Critical (attracts penalty)
3. Neutral Current &gt; 10% of phase current average — Warning
4. Fire_Risk_Level = &#39;WARNING&#39; or &#39;HIGH&#39; — Flag immediately
Task:
1. Write a Python script that scans the data log for power quality issues
2. Generate an alert log with: timestamp, device, alert type, severity, actual value
3. Summarize: total alerts by type, alerts by device, most frequent issues
4. Bonus: Suggest possible causes for observed patterns
Evaluation Criteria:
• Correct threshold implementation
• Comprehensive alert capture
• Useful summary and insights
• Code quality and documentation

In [30]:
import pandas as pd
path = "vireon_sample_data.tsv"
df = pd.read_csv(path,sep="\t")
df.head()

,Timestamp,Date,Time,Hour,Device_ID,Location,Meter_Serial,Model_Number,ToD_Period,Rate_Rs_kWh,...,Daily_Energy_Wh,PF_Rebate_Pct,Effective_Rate,Daily_Cost_Rs,Carbon_kg_CO2,Pi_Temp_C,Pi_CPU_Pct,Pi_Mem_Pct,Pi_Disk_Pct,Pi_Uptime_Hrs
0,2026-02-12 22:14:48,2026-02-12,22:14:48,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,477920,8,7.70,3054.29,41134.2,37.3,0.0,6.6,24.6,9.3
1,2026-02-12 22:18:11,2026-02-12,22:18:11,22,vireon-cortex-cl01-mu-02,Shed_02,24057940XXX,4405,PEAK,8.37,...,44209,0,8.37,254.37,3349.9,47.2,0.0,11.0,25.4,1.1
2,2026-02-12 22:20:20,2026-02-12,22:20:20,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,478044,7,7.78,3055.25,41134.3,38.4,0.0,6.7,24.6,9.4
3,2026-02-12 22:23:41,2026-02-12,22:23:41,22,vireon-cortex-cl01-mu-02,Shed_02,24057940XXX,4405,PEAK,8.37,...,44209,0,8.37,254.37,3349.9,47.7,0.0,11.0,25.4,1.2
4,2026-02-12 22:25:49,2026-02-12,22:25:49,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,478172,8,7.70,3056.24,41134.4,38.4,0.0,6.7,24.6,9.5


In [31]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['timestamp', 'date', 'time', 'hour', 'device_id', 'location',
       'meter_serial', 'model_number', 'tod_period', 'rate_rs_kwh',
       'watts_total', 'watts_r', 'watts_y', 'watts_b', 'kw_total', 'va_total',
       'va_r', 'va_y', 'va_b', 'kva_total', 'pf_avg', 'pf_r', 'pf_y', 'pf_b',
       'vll_avg', 'v_ry', 'v_yb', 'v_br', 'vln_avg', 'v_r', 'v_y', 'v_b',
       'current_total', 'current_r', 'current_y', 'current_b',
       'neutral_current_a', 'fire_risk_level', 'frequency_hz', 'energy_wh',
       'energy_kwh', 'energy_vah', 'energy_vah_derived',
       'voltage_unbalance_pct', 'current_unbalance_pct', 'load_pct',
       'run_hours', 'max_demand_kva', 'max_demand_kw', 'session_energy_wh',
       'daily_energy_wh', 'pf_rebate_pct', 'effective_rate', 'daily_cost_rs',
       'carbon_kg_co2', 'pi_temp_c', 'pi_cpu_pct', 'pi_mem_pct', 'pi_disk_pct',
       'pi_uptime_hrs'],
      dtype='str')

Key columns in the dataset:
• Timestamp, Date, Time, Hour — When the reading was taken
• Device_ID, Location — Which meter/shed the reading is from
• ToD_Period, Rate_Rs_kWh — Time-of-Day tariff (PEAK/OFFPEAK) and rate
• Watts_Total, kW_Total, kVA_Total — Power consumption
• V_R, V_Y, V_B, VLL_Avg, VLN_Avg — Voltage readings (3-phase)
• Current_R, Current_Y, Current_B, Current_Total — Current readings
• PF_Avg, PF_R, PF_Y, PF_B — Power factor readings
• Energy_kWh, Daily_Energy_Wh — Cumulative and daily energy
• Fire_Risk_Level — Derived safety indicator (NORMAL/HIGH/WARNING)
• Voltage_Unbalance_Pct, Current_Unbalance_Pct — Power quality metrics
• Pi_Temp_C, Pi_CPU_Pct, Pi_Mem_Pct — Edge device health

voltage_unbalance_pct > 2 , voltage_unbalance_pct > 5
pf_avg < 0.9 , pf_avg < 0.85
neutral_current_a > 0.1*avg_ph
fire_risk_level - HIGH/WARNING - flag

In [32]:
avg_ph = (df["current_r"] + df["current_y"] + df["current_b"]) / 3
avg_ph

0      1.903333
1      0.000000
2      1.856667
3      0.000000
4      1.903333
        ...    
57     3.640000
58    23.640000
59     3.810000
60    15.776667
61     7.366667
Length: 62, dtype: float64

In [33]:
v1 = df.loc[(df["voltage_unbalance_pct"] > 2) & df["voltage_unbalance_pct"] <= 5, ["timestamp", "location", "voltage_unbalance_pct"]].assign(alert = "Voltage Unbalance", type = "Warning")
v2 = df.loc[df["voltage_unbalance_pct"] > 5, ["timestamp", "location", "voltage_unbalance_pct"]].assign(alert = "Voltage Unbalance", type = "Critical")

pf1 = df.loc[(df["pf_avg"] < 0.9)& df["pf_avg"] >= 0.85,  ["timestamp", "location", "pf_avg"]].assign(alert = "Power Factor", type = "Warning")
pf2 = df.loc[df["pf_avg"]< 0.85, ["timestamp", "location", "pf_avg"]].assign(alert = "Power Factor", type = "Critical")

nc = df.loc[df["neutral_current_a"] > 0.1*avg_ph, ["timestamp", "location", "neutral_current_a"]].assign(alert = "Neutral Current", type = "Warning")

fr1 = df.loc[df["fire_risk_level"] == "WARNING", ["timestamp", "location", "fire_risk_level"]].assign(alert = "Fire Risk Level", type = "Warning")
fr2 = df.loc[df["fire_risk_level"] == "HIGH", ["timestamp", "location", "fire_risk_level"]].assign(alert = "Fire Risk Level", type = "Critical")

alert=[v1,v2,pf1,pf2,nc,fr1,fr2]

In [34]:
alert = pd.concat([v1,v2,pf1,pf2,nc,fr1,fr2], ignore_index=True)
print(alert.shape)

(155, 8)


In [35]:
alert["alert"].value_counts()

alert
Voltage Unbalance    62
Neutral Current      52
Fire Risk Level      31
Power Factor         10
Name: count, dtype: int64

In [36]:
alert["location"].value_counts()

location
Shed_01    91
Shed_02    64
Name: count, dtype: int64

In [37]:
print("Max alerts from ", alert["alert"].value_counts().idxmax())

Max alerts from  Voltage Unbalance
